# 01 · Baselines and how to validate

Two questions before any real model:

1. **What score must a model beat?** Simple rules a collections team could use without a model (runs **B-01 … B-05**).
2. **Which validation method gives a trustworthy estimate?** Single 80/20 split vs 5-fold vs 10-fold vs repeated
   5×5 vs leave-one-out — on the full data and on a *real-sized* sample of 60 rows (runs **V-01 … V-06**).

Every run is logged to `reports/phase2/experiments/results.csv` and appears on the shared leaderboard.

`DESIGN` picks the synthetic training rows (notebook 00, section 6): `"fixed"` = one row per patient with an open
balance on the scoring date (recommended for the pre-call model), `"call"` = one row per call.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from salud_pipeline.phase2.propensity import modeling as M
from salud_pipeline.phase2.propensity import dictionary as D
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
DESIGN = "fixed"   # or "call"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
print(f"dev (synthetic, 80% for experiments): {len(dev):,} rows, {dev.guarantor_id.nunique():,} patients, pay rate {dev.paid_30d.mean():.2f}")
print(f"holdout (synthetic, 20% held back, used once in notebook 11): {len(holdout):,} rows")
print(f"real (usable, final test only): {len(real)} rows, pay rate {real.paid_30d.mean():.2f}")

## 1. Baselines (pre-call target, grouped 5×5 CV)

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import roc_auc_score

class Rule(BaseEstimator, ClassifierMixin):
    """A fixed scoring rule (no learning). score(X) -> higher = more likely to pay."""
    def __init__(self, score=None): self.score = score
    def fit(self, X, y): self.classes_ = np.array([0, 1]); return self
    def predict_proba(self, X):
        s = np.asarray(self.score(X), float); s = (s - s.min()) / (np.ptp(s) or 1)
        return np.c_[1 - s, s]

class BestSingleFeature(BaseEstimator, ClassifierMixin):
    """Picks the single strongest feature on the training fold only (no peeking at the test fold)."""
    def fit(self, X, y):
        best = {c: roc_auc_score(y, X[c].fillna(X[c].median())) for c in X.columns}
        self.feature_ = max(best, key=lambda c: abs(best[c] - .5)); self.sign_ = 1 if best[self.feature_] >= .5 else -1
        self.classes_ = np.array([0, 1]); return self
    def predict_proba(self, X):
        s = self.sign_ * X[self.feature_].fillna(X[self.feature_].median()).to_numpy(float)
        s = (s - s.min()) / (np.ptp(s) or 1); return np.c_[1 - s, s]

rng = np.random.default_rng(M.SEED)
base_runs = [
    ("B-01", Rule(lambda X: rng.random(len(X))), "Random scores (sanity check)"),
    ("B-02", Rule(lambda X: (X["n_payments_365d"] > 0).astype(float)), "Rule: paid in the last year → likely"),
    ("B-03", Rule(lambda X: (X["n_payments_365d"] > 0) * 1000 - X["days_since_last_payment"].fillna(999)), "Rule: paid before, most recent payment first"),
    ("B-04", BestSingleFeature(), "Best single feature (chosen inside each fold)"),
]
for rid, est, desc in base_runs:
    M.run_experiment(rid, est, dev, "FS_TRANSFER", notebook="01", description=desc)
dev_o = pd.concat([dev, M.oracle_traits(dev, WORLD)], axis=1)
M.run_experiment("B-05", M.pipeline(M.preprocess("z"), M.logreg(C=1.0)), dev_o, ["ability", "willingness", "hardship"],
                 notebook="01", description="ORACLE: hidden traits (not observable)");

In [ ]:
M.run_experiment("REF-LR", M.pipeline(M.preprocess("log_z"), M.logreg(C=1.0)), dev, "FS_TRANSFER", notebook="01",
                 description="Reference model: L2 logistic regression, log1p + z-score, FS_TRANSFER")
board = M.leaderboard(notebook="01")
board[["run_id", "description", "auc", "auc_sd", "brier", "top20", "n_features"]]

In [ ]:
r = pd.read_csv(M.RESULTS); r = r[r.notebook.astype(str) == "01"]
order = board.run_id.tolist()
plt.figure(figsize=(9, 3.5)); sns.boxplot(data=r, y="run_id", x="auc", order=order, color="#8DA0CB")
plt.axvline(.5, ls="--", c="grey"); plt.title("AUC across 25 folds"); plt.xlabel("AUC"); plt.ylabel(""); plt.show()

## 2. Which validation method? (V-01 … V-05)

The same reference model is scored by each method, **many times with different random seeds**. A good method gives
estimates that are both **accurate** (close to the model's true performance) and **stable** (little spread).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, LeaveOneGroupOut
from sklearn.base import clone
REF = M.pipeline(M.preprocess("log_z"), M.logreg(C=1.0))
X_all, y_all = M.xy(dev, M.FS_TRANSFER)
g_all = dev.guarantor_id.to_numpy()

def holdout_est(X, y, g, seed):
    tr, te = next(GroupShuffleSplit(1, test_size=.2, random_state=seed).split(X, y, g))
    if len(set(y[te])) < 2: return np.nan
    return roc_auc_score(y[te], clone(REF).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1])

def kfold_est(X, y, g, seed, k, repeats=1):
    df_ = pd.DataFrame({"paid_30d": y, "guarantor_id": g, "snapshot_id": np.arange(len(y)).astype(str)})
    aucs = []
    for s in M.cv_splits(df_, k=k, repeats=repeats, seed=seed):
        if len(set(y[s.test])) == 2:
            aucs.append(roc_auc_score(y[s.test], clone(REF).fit(X.iloc[s.train], y[s.train]).predict_proba(X.iloc[s.test])[:, 1]))
    return np.mean(aucs)

def logo_est(X, y, g):   # leave-one-patient-out, predictions pooled then one AUC
    p = np.zeros(len(y))
    for tr, te in LeaveOneGroupOut().split(X, y, g):
        p[te] = clone(REF).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y, p)

import time
rows = []
for name, fn, seeds in [("V-01 single 80/20 split", lambda s: holdout_est(X_all, y_all, g_all, s), range(25)),
                        ("V-02 5-fold", lambda s: kfold_est(X_all, y_all, g_all, s, 5), range(25)),
                        ("V-03 10-fold", lambda s: kfold_est(X_all, y_all, g_all, s, 10), range(10)),
                        ("V-04 repeated 5×5 (chosen)", lambda s: kfold_est(X_all, y_all, g_all, s, 5, 5), range(5))]:
    t0 = time.perf_counter(); est = [fn(s) for s in seeds]; secs = (time.perf_counter() - t0) / len(est)
    rows.append({"method": name, "estimates": len(est), "mean AUC": np.nanmean(est), "spread (SD across seeds)": np.nanstd(est),
                 "seconds per estimate": secs})
full = pd.DataFrame(rows)
full

### At real-data size (n = 60)
The real test set has 58 rows. Draw 60-row samples from the synthetic data many times; for each, estimate AUC with
each method **using only those 60 rows**, and compare with the model's *true* performance (the same 60-row model
scored on 1,000 untouched rows). This shows how wrong each method can be at Salud's data size.

In [ ]:
rng = np.random.default_rng(M.SEED)
big_test = dev.sample(min(1000, len(dev) // 2), random_state=1)
pool = dev.drop(big_test.index)
Xb, yb = M.xy(big_test, M.FS_TRANSFER)
recs = []
for i in range(60):
    s = pool.sample(60, random_state=100 + i)
    Xs, ys = M.xy(s, M.FS_TRANSFER); gs = s.guarantor_id.to_numpy()
    if ys.sum() < 8 or (len(ys) - ys.sum()) < 8: continue
    true = roc_auc_score(yb, clone(REF).fit(Xs, ys).predict_proba(Xb)[:, 1])
    for name, est in [("single 80/20", holdout_est(Xs, ys, gs, i)), ("5-fold", kfold_est(Xs, ys, gs, i, 5)),
                      ("repeated 5×5", kfold_est(Xs, ys, gs, i, 5, 5)), ("leave-one-patient-out", logo_est(Xs, ys, gs))]:
        recs.append({"method": name, "estimate": est, "true": true, "error": est - true})
small = pd.DataFrame(recs)
tab = small.groupby("method").agg(mean_estimate=("estimate", "mean"), mean_true=("true", "mean"),
                                  bias=("error", "mean"), typical_error=("error", lambda e: np.sqrt(np.nanmean(e ** 2))))
tab.sort_values("typical_error")

In [ ]:
plt.figure(figsize=(9, 3.5)); sns.boxplot(data=small, y="method", x="error", color="#FC8D62")
plt.axvline(0, c="k", lw=1); plt.title("Estimate minus true AUC, 60-row samples"); plt.xlabel("error"); plt.ylabel(""); plt.show()

## 3. What if we forget to group by patient? (V-06)

In [ ]:
grouped = M.run_experiment("V-04", REF, dev, "FS_TRANSFER", notebook="01", description="Grouped by patient (correct)")
ungrouped = M.run_experiment("V-06", REF, dev, "FS_TRANSFER", notebook="01", description="NOT grouped (leaks patients)",
                             splits=M.cv_splits(dev, grouped=False))
print(f"Inflation from not grouping: {ungrouped.auc.mean() - grouped.auc.mean():+.3f} AUC")

## Findings (Salud-tuned world)
- **Baselines:**

  | Run | AUC |
  |---|---|
  | Random (B-01) | 0.508 |
  | Paid in the last year (B-02) | 0.701 |
  | **Paid before, most recent first (B-03)** | **0.843** |
  | Best single feature (B-04) | 0.848 |
  | Reference L2 logistic regression (REF-LR) | 0.867 |

  Most of the signal is payment history, as in Salud's data.
  A model has to beat about 0.85 to be worth more than a rule.
- **Validation:** the repeated 5×5 estimate has the smallest spread across seeds (SD 0.0003, against 0.023 for a single split). It costs about 0.6 s per estimate. Keep it.
- **Leakage check:** not grouping inflates the AUC by only +0.001 here, because the fixed design has one row per patient.
  Grouping still matters for the call design (several rows per patient), and it stays on everywhere.